In [ ]:
%pip install torch transformers

In [1]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

In [3]:
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='featext2'
)

                                                    url  type  \
0                https://reviewe-014035.firebaseapp.com     0   
1                                     http://www.0up.ir     0   
2                        http://www.websitetosubmit.com     0   
3                  http://muj22389057436506320965.rf.gd     0   
4                                 http://www.orino.info     0   
...                                                 ...   ...   
3599                       http://members.shaw.ca/co-bc     0   
3600    http://www.theatrehistory.com/plays/salmon.html     0   
3601                  https://teleofficialxx2.pages.dev     1   
3602  http://43.153.207.103/v3/signin/identifier?dsh...     1   
3603                    http://bt-109938.weeblysite.com     1   

      content_redirects  content_len_html  content_len_text  \
0                    -1                -1                -1   
1                    -1                -1                -1   
2                    -1       

In [4]:
duplicated_urls_count = dataloader.df['url'].duplicated().sum()
print(f"Number of duplicated URLs: {duplicated_urls_count}")

dataloader.df.drop_duplicates(subset='url', inplace=True)

Number of duplicated URLs: 0


In [5]:
print(f"Number of null DOM screenshots: {dataloader.df['dom_screenshot_url'].isna().sum()}")
dataloader.df.dropna(subset=['dom_screenshot_url'], how='all', inplace=True)

Number of null DOM screenshots: 0


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
print(dataloader.df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 43604 entries, 0 to 3603
Data columns (total 68 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   url                                    43604 non-null  object 
 1   type                                   43604 non-null  int64  
 2   content_redirects                      43604 non-null  int64  
 3   content_len_html                       43604 non-null  int64  
 4   content_len_text                       43604 non-null  int64  
 5   content_len_links                      43604 non-null  int64  
 6   content_len_mail_usage_forms           43604 non-null  int64  
 7   content_meta_script_link_percentage    43604 non-null  object 
 8   content_mouseover_changes              43604 non-null  int64  
 9   content_right_click_disabled           43604 non-null  int64  
 10  content_keyboard_shortcuts_disabled    43604 non-null  int64  
 11  con

In [10]:
import re
from urllib.parse import urlparse

# Sample the first 1000 URLs (or fewer if the dataset is smaller)
sample_size = min(50000, len(dataloader.df))
url_sample = dataloader.df['url'].sample(n=sample_size, random_state=42)

# Initialize counters
http_count = 0
https_count = 0
other_scheme_count = 0
www_count = 0
ip_count = 0
subdomain_count = 0

# Regex patterns
ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
subdomain_pattern = r'(?:http[s]?://)?(?:www\.)?([a-zA-Z0-9-]+\.)*[a-zA-Z0-9-]+\.[a-zA-Z]+'

# Analyze URLs
for url in url_sample:
    parsed_url = urlparse(url)
    
    # Check scheme
    if parsed_url.scheme == 'http':
        http_count += 1
    elif parsed_url.scheme == 'https':
        https_count += 1
    else:
        other_scheme_count += 1
    
    # Check for www
    if parsed_url.netloc.startswith('www.'):
        www_count += 1
    
    # Check for IP addresses
    if re.match(ip_pattern, parsed_url.netloc):
        ip_count += 1
    
    # Check for subdomains
    if re.match(subdomain_pattern, url):
        subdomain_count += 1

# Print results
print(f"Total URLs analyzed: {sample_size}")
print(f"HTTP URLs: {http_count} ({http_count/sample_size*100:.2f}%)")
print(f"HTTPS URLs: {https_count} ({https_count/sample_size*100:.2f}%)")
print(f"Other scheme URLs: {other_scheme_count} ({other_scheme_count/sample_size*100:.2f}%)")
print(f"URLs with 'www': {www_count} ({www_count/sample_size*100:.2f}%)")
print(f"URLs with IP addresses: {ip_count} ({ip_count/sample_size*100:.2f}%)")
print(f"URLs with subdomains: {subdomain_count} ({subdomain_count/sample_size*100:.2f}%)")

# Print some example URLs
print("\nExample URLs:")
print(url_sample.head(10).to_string())

# Additional regex patterns that might be useful
special_char_pattern = r'[!@#$%^&*(),.?":{}|<>]'
number_in_domain_pattern = r'://[^/]*\d'
long_subdomain_pattern = r'://([a-zA-Z0-9-]+\.){3,}'

# Count URLs matching these patterns
special_char_count = url_sample.str.contains(special_char_pattern).sum()
number_in_domain_count = url_sample.str.contains(number_in_domain_pattern).sum()
long_subdomain_count = url_sample.str.contains(long_subdomain_pattern).sum()

print(f"\nURLs with special characters: {special_char_count} ({special_char_count/sample_size*100:.2f}%)")
print(f"URLs with numbers in domain: {number_in_domain_count} ({number_in_domain_count/sample_size*100:.2f}%)")
print(f"URLs with long subdomains: {long_subdomain_count} ({long_subdomain_count/sample_size*100:.2f}%)")

Total URLs analyzed: 43604
HTTP URLs: 8644 (19.82%)
HTTPS URLs: 34960 (80.18%)
Other scheme URLs: 0 (0.00%)
URLs with 'www': 38907 (89.23%)
URLs with IP addresses: 52 (0.12%)
URLs with subdomains: 43554 (99.89%)

Example URLs:
3966                                  http://www.start.lc
8899                                 http://www.p2020.xyz
9095                    https://www.best-infographics.com
95                                     https://www.pgu.sk
8090                                http://coinbasedex.me
2349                     https://www.secondcityhockey.com
3546                          https://www.quirkycrate.com
6556    https://derickcarvalho.github.io/loginpage-net...
1470                          https://www.deafsparrow.com
1707                         https://www.city.otaru.lg.jp

URLs with special characters: 43604 (100.00%)
URLs with numbers in domain: 4401 (10.09%)
URLs with long subdomains: 7064 (16.20%)


/var/folders/lz/54h48fsx6294gn68v0q9pp4m0000gn/T/ipykernel_20405/2792898155.py:65: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  long_subdomain_count = url_sample.str.contains(long_subdomain_pattern).sum()


In [7]:
class_distribution = dataloader.df['type'].value_counts()
print("\nClass Distribution:")
print(class_distribution)


Class Distribution:
1    31171
0    12433
Name: type, dtype: int64


In [21]:
dataloader.df.iloc[1]['dom_screenshot_url']

'https://urlscan.io/screenshots/d1f70d56-7842-4356-87e7-0bca8b1d4556.png'

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
urls = dataloader.df['url'].tolist()
labels = dataloader.df['type'].tolist()

In [ ]:
len(urls)

In [ ]:
# Using a pre-trained tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize URLs
def tokenize_urls(urls):
    return tokenizer(urls, padding=True, truncation=True, max_length=512, return_tensors='pt')

class PhishingDataset(Dataset):
    def __init__(self, encodings, labels):
        # Encodings are expected to be a dict where values are already tensors
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Directly use the tensor slices without re-wrapping them into new tensors
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return self.labels.size(0)

# Assume URLs and labels are already defined above
encoded_inputs = tokenize_urls(urls)

# Convert the labels to a tensor outside of the dataset initialization to ensure it is properly managed
labels_tensor = torch.tensor(labels, dtype=torch.long)

# Create the dataset
dataset = PhishingDataset(encoded_inputs, labels_tensor)

In [ ]:
# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all layers except the classifier to speed up training (optional)
for name, param in model.named_parameters():
    if 'classifier' not in name:  # Freeze layers other than the classifier
        param.requires_grad = False

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=dataset,         # training dataset
)

trainer.train()

In [ ]:
trainer.evaluate()